In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex Agent Builder and Dialogflow CX Evaluations (+multilingual option)
In this notebook, we will show you how to:
1. Build a new Evaluation Dataset for your Agent.
2. Run the Evaluations to get Quality Metrics
3. Push the Results to Google Sheets for reporting.


## Prerequisites
- Existing Agent Builder or DFCX Agent w/ or w/out Tool calling.
<br>

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/vertex_agents_evals.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/colab-logo-32px.png" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/vertex_agents_evals.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/github-logo-32px.png" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/vertex_agents_evals.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>

In [3]:
!pip install dfcx-scrapi>=2.0.0
# The language parameter was introduced in 2.0.0, refer to https://github.com/GoogleCloudPlatform/dfcx-scrapi/pull/291

import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    from google.auth import default

    auth.authenticate_user()
    credentials, _ = default()
else:
    # Otherwise, attempt to discover local credentials as described in
    # https://cloud.google.com/docs/authentication/application-default-credentials
    pass


# Evaluation Dataset Format

Collecting and evaluating multi-turn chat data can be complex, so we have devised template that you can follow to make it easy and scalable.

The evaluation dataset must be in a tabular format and contain the following columns:
- `eval_id` 
  - _Unique identifier of a conversation, which must be the same for each row that is part of the same conversation_
- `action_id` 
  - _Index of the specific action for the current conversation._
  - _This is used to track and pair responses during inference time._
- `action_type` 
  - _The specific action for this turn._
  - _There are currently 4 supported Action Types: `User Utterance`, `Agent Response`, `Playbook Invocation`, `Tool Invocation`_
- `action_input`
  - _The input for this specific action_type._
  - _Based on the specified action_type, this could be the expected user utterance or agent response, the expected Playbook name, or the expected Tool name._
- `action_input_parameters`
  - When `Playbook Invocation` or `Tool Invocation` is selected as the `action_type`, this refers to the payload of information that is expected to be sent with that invocation._
  - _For example, a JSON payload of key/value pairs called with the tool._
- `tool_action`
  - _This field is only used when `Tool Invocation` is chosen as the `action_type`._
  - _This allows us to run evaluations on whether the Tool call chose the correct internal action (if more than one exists)_
- `language_code` **(Optional Metadata)**
  - _The language code for this conversation. This column is useful for organizing your dataset and processing multilingual evaluations._
  - _Use standard language codes as supported by Dialogflow CX [Complete list of supported language codes](https://cloud.google.com/dialogflow/cx/docs/reference/language)._
  - _**Important**: This column is metadata. To evaluate conversations in specific languages, you must:_
    - _Either pass a `language_code` parameter to evaluate all conversations in that language_
    - _Or split your dataframe by this column and process each language separately (see Option 3 in Predict and Evaluate)_
  - _If not specified, the default language code "en" (English) is used._

---

An example for the queryset can be seen in this table:

| eval_id | action_id | action_type | action_input | action_input_parameters | tool_action | language_code | notes |
|---|---|---|---|---|---|---|---|
| travel-ai-001 | 1 | User Utterance | Paris |  |  | en |  |
| travel-ai-001 | 2 | Playbook Invocation | Travel Inspiration |  |  | en |  |
| travel-ai-001 | 3 | Agent Response | Paris is a beautiful city! Here are a few things you might enjoy doing there:<br><br>Visit the Eiffel Tower<br>Take a walk along the Champs-Élysées<br>Visit the Louvre Museum<br>See the Arc de Triomphe<br>Take a boat ride on the Seine River |  |  | en |  |
| travel-ai-002 | 1 | User Utterance | Je veux aller à Montréal |  |  | fr-CA |  |
| travel-ai-002 | 2 | Playbook Invocation | Travel Inspiration |  |  | fr-CA |  |
| travel-ai-002 | 3 | Agent Response | Je serais heureux de vous aider à trouver un hôtel à Montréal |  |  | fr-CA |  |
| travel-ai-002 | 4 | User Utterance | Du 1er au 10 juin |  |  | fr-CA |  |
| travel-ai-002 | 5 | Playbook Invocation | Book Hotel |  |  | fr-CA |  |
| travel-ai-002 | 6 | Tool Invocation | hotel_tool | {'city': 'Montreal', 'num_results': 10} | hotel_search | fr-CA |  |
---

# Data Loading
The preferred method for data loading is to use Google Sheets.  
However you can also manually build your dataset as a Pandas Dataframe, or load from CSV, BQ, etc. as needed.

## Option 1 - From Google Sheets

In [ ]:
from dfcx_scrapi.tools.evaluations import DataLoader

data = DataLoader()

sheet_name = "[TEMPLATE] Vertex Agent Evals Dataset Format"
sheet_tab = "golden-agent-evals"

sample_df = data.from_google_sheets(sheet_name, sheet_tab)

## Option 2 - From Local CSV

In [ ]:
from dfcx_scrapi.tools.evaluations import DataLoader

data = DataLoader()

csv_file_path = ""

sample_df = data.from_csv(csv_file_path)

# If your CSV includes a language_code column, it will be preserved as metadata
# To evaluate conversations in specific languages, use Option 1 or 2 in the 
# "Predict and Evaluate" section, or process by language (Option 3)


## Option 3 - Manual Loading

In [ ]:
import pandas as pd
from dfcx_scrapi.tools.evaluations import DataLoader

data = DataLoader()

# Include language_code as optional metadata column
INPUT_SCHEMA_REQUIRED_COLUMNS = ['eval_id', 'action_id', 'action_type', 'action_input', 'action_input_parameters', 'tool_action', 'language_code', 'notes']

sample_df = pd.DataFrame(columns=INPUT_SCHEMA_REQUIRED_COLUMNS)

# English examples
sample_df.loc[0] = ["travel-ai-001", 1, "User Utterance", "Paris", "", "", "en", ""]
sample_df.loc[1] = ["travel-ai-001", 2, "Playbook Invocation", "Travel Inspiration", "", "", "en", ""]
sample_df.loc[2] = ["travel-ai-001", 3, "Agent Response", "Paris is a beautiful city! Here are a few things you might enjoy doing there:\n\nVisit the Eiffel Tower\nTake a walk along the Champs-Élysées\nVisit the Louvre Museum\nSee the Arc de Triomphe\nTake a boat ride on the Seine River", "", "", "en", ""]

# French-Canadian examples (same conversation ID pattern can be used)
sample_df.loc[3] = ["travel-ai-002", 1, "User Utterance", "Je veux aller à Montréal", "", "", "fr-CA", ""]
sample_df.loc[4] = ["travel-ai-002", 2, "Playbook Invocation", "Travel Inspiration", "", "", "fr-CA", ""]
sample_df.loc[5] = ["travel-ai-002", 3, "Agent Response", "Je serais heureux de vous aider à trouver un hôtel à Montréal", "", "", "fr-CA", ""]

sample_df = data.from_dataframe(sample_df)


# Metrics
For multi-turn chat Agents w/ or w/out tool calling, there are currently 2 metrics supported:
1. `Response Similarity`, this performs a Semantic Similarity comparison between the expected "golden" response and the actual "predicted" response
2. `Tool Call Quality`, this performs and EXACT_MATCH on 2 components of the Tool call
    - Tool Name, i.e. was the correct Tool invoked
    - Tool Action, i.e. for the given Tool, was the correct Action / Endpoint invoked

Other metrics like `UrlMatch`, `Faithfulness`, `Answer Correctness`, `Context Recall` etc. will be supported in the future.

## Language Support
All metrics support evaluation across different languages via the `language_code` parameter:
- **Default language**: `"en"` (English) if `language_code` is not specified
- **Global override**: Pass `language_code` parameter to evaluate all queries in a specific language (e.g., `language_code="fr-CA"`)
- **Note**: The `language_code` column in your dataframe is metadata. To use it, you must explicitly process each language group separately (see Option 3 in the Predict and Evaluate section)


In [2]:
from dfcx_scrapi.tools.evaluations import Evaluations

# [1] Define your Agent ID here
agent_id = "projects/your-project/locations/us-central1/agents/11111-2222-33333-44444" # Example Agent

# [2] Instantiate Evals class w/ Metrics
evals = Evaluations(agent_id, metrics=["response_similarity", "tool_call_quality"])

# Predict and Evaluate
In this step, we will run all of the queries against the Agent that is being evaluated.  
Once the queries are returned, we will then compute all of the metrics.

## Language Code Handling

The `run_query_and_eval()` method signature is: `run_query_and_eval(df, language_code="en")`

Thre are three approaches for handling language codes:



### Option 1: Default to English (Recommended for single-language datasets)
If your dataset contains only one language or you want to evaluate everything in English by default:

In [ ]:
# OPTION 1: Default to English (ignores language_code column)
eval_results = evals.run_query_and_eval(sample_df.head(10))

print(f"Average Similarity {eval_results.similarity.mean()}")
print(f"Average Tool Call Quality {eval_results.tool_name_match.mean()}")

### Option 2: Override Default Language (Recommended for single-language evaluation)
If you want all queries evaluated in a specific language other than English:

In [ ]:
eval_results = evals.run_query_and_eval(sample_df.head(10), language_code="fr-CA")

print(f"Average Similarity {eval_results.similarity.mean()}")
print(f"Average Tool Call Quality {eval_results.tool_name_match.mean()}")

### Option 3: Split by Language and Process Separately (For multilingual datasets)
If your dataset contains conversations in **multiple different languages**, you must split by language and process each separately:

In [ ]:
import pandas as pd

all_results = []
for language_code, language_df in sample_df.groupby('language_code'):
    print(f"Processing language: {language_code} ({len(language_df)} rows)...")
    # Pass the language_code parameter explicitly
    results = evals.run_query_and_eval(language_df.head(10), language_code=language_code)
    results['language'] = language_code  # Add language label to results
    all_results.append(results)

# Combine all results
eval_results = pd.concat(all_results, ignore_index=True)

# Restore original order if needed (sort by eval_id or another identifier)
# eval_results = eval_results.sort_values('eval_id').reset_index(drop=True)

# Analyze by language
print("\nResults by Language:")
for language in eval_results['language'].unique():
    lang_results = eval_results[eval_results['language'] == language]
    print(f"{language}: Avg Similarity = {lang_results.similarity.mean():.4f}")
    print(f"{language}: Avg Tool Call Quality = {lang_results.tool_name_match.mean():.4f}")


In [ ]:
# You can inspect the results as needed
eval_results.head()

# Write to Google Sheets
Storing the evaluation results to Google Sheets can be done with the following snippets.

In future revisions, we will add export to other format including `csv`, `bigquery`, etc.

In [ ]:
from dfcx_scrapi.tools.evaluations import DataLoader

data = DataLoader()

data.write_eval_results_to_sheets(eval_results, sheet_name, results_tab="latest_results")
data.append_test_results_to_sheets(eval_results, sheet_name, summary_tab="reporting")

# Ending and Wrap-Up

In this notebook, we've shown how to programmatically Evaluate your Agent Builder or Dialogflow CX Agent.

For more information, see:
- [Vertex AI Agents](https://cloud.google.com/dialogflow/vertex/docs/concept/agents)
- [Dialogflow CX](https://cloud.google.com/dialogflow/cx/docs)